In [3]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import ta  # Technical Analysis library (pip install ta)

df = pd.read_csv('../data/processed/merged/merged_data.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df.head()

,Date,gold_close,gold_high,gold_low,gold_open,gold_vol,dxy_close,vix_close,yield_close,sp500_close,oil_close
0,2015-01-02,1186.000000,1194.500000,1169.500000,1184.000000,138,91.080002,17.790001,2.123,2058.199951,52.689999
1,2015-01-05,1203.900024,1206.900024,1180.099976,1180.300049,470,91.379997,19.920000,2.039,2020.579956,50.040001
2,2015-01-06,1219.300049,1220.000000,1203.500000,1203.500000,97,91.500000,21.120001,1.963,2002.609985,47.930000
3,2015-01-07,1210.599976,1219.199951,1210.599976,1219.199951,29,91.889999,19.309999,1.954,2025.900024,48.650002
4,2015-01-08,1208.400024,1215.699951,1206.300049,1207.000000,92,92.370003,17.010000,2.016,2062.139893,48.790001


# Feature Engineering

ในส่วนนี้จะสร้างฟีเจอร์ 3 กลุ่มหลัก ได้แก่
- Lag Features (ย้อนหลัง 1, 5, 10, 20 วัน)
- Technical Indicators (RSI, MACD, Bollinger Bands, ATR)
- Macro %change (DXY, VIX, Yield, SP500, Oil)

และสร้าง Target Variable สำหรับงาน Regression และ Classification

## 1. Lag Features

สร้างฟีเจอร์ราคาย้อนหลัง (lag) ของแต่ละสินทรัพย์ ได้แก่ gold, dxy, vix, yield, sp500, oil โดยใช้ช่วงเวลา 1, 5, 10, 20 วัน

In [9]:
# สร้าง lag ทุกตัวก่อน
for col in ['gold_close','dxy_close','vix_close','yield_close','sp500_close','oil_close']:
    for lag in [1, 5, 10, 20]:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)
# ค่อย dropna หลังจากสร้างฟีเจอร์ทั้งหมด
df = df.dropna().reset_index(drop=True)
df.head()

,Date,gold_close,gold_high,gold_low,gold_open,gold_vol,dxy_close,vix_close,yield_close,sp500_close,...,yield_close_lag10,yield_close_lag20,sp500_close_lag1,sp500_close_lag5,sp500_close_lag10,sp500_close_lag20,oil_close_lag1,oil_close_lag5,oil_close_lag10,oil_close_lag20
0,2015-03-03,1204.000000,1212.000000,1195.000000,1201.500000,51,95.379997,13.86,2.122,2107.780029,...,2.145,1.673,2117.389893,2115.479980,2100.340088,2020.849976,49.590000,49.279999,53.529999,49.570000
1,2015-03-04,1200.599976,1206.900024,1198.000000,1204.800049,27,95.970001,14.23,2.123,2098.530029,...,2.066,1.780,2107.780029,2113.860107,2099.679932,2050.030029,50.520000,50.990002,52.139999,53.049999
2,2015-03-05,1195.900024,1205.400024,1195.900024,1201.699951,29,96.379997,14.04,2.112,2101.040039,...,2.113,1.797,2098.530029,2110.739990,2097.449951,2041.510010,51.529999,48.169998,51.160000,48.450001
3,2015-03-06,1164.099976,1195.699951,1163.699951,1195.000000,168,97.620003,15.20,2.240,2071.260010,...,2.133,1.815,2101.040039,2104.500000,2110.300049,2062.520020,50.759998,49.759998,50.340000,50.480000
4,2015-03-09,1166.400024,1174.199951,1166.300049,1169.400024,45,97.589996,15.06,2.195,2079.429932,...,2.059,1.938,2071.260010,2117.389893,2109.659912,2055.469971,49.610001,49.590000,49.450001,51.689999


## 2. Technical Indicators

สร้างฟีเจอร์ทางเทคนิค เช่น RSI, MACD, Bollinger Bands, ATR จากราคาทองคำ (gold_close)

In [13]:

import ta 

df['gold_rsi'] = ta.momentum.RSIIndicator(df['gold_close'], window=14).rsi()
df['gold_macd'] = ta.trend.MACD(df['gold_close']).macd()
df['gold_macd_signal'] = ta.trend.MACD(df['gold_close']).macd_signal()
df['gold_macd_diff'] = ta.trend.MACD(df['gold_close']).macd_diff()
bb = ta.volatility.BollingerBands(df['gold_close'], window=20)
df['gold_bb_high'] = bb.bollinger_hband()
df['gold_bb_low'] = bb.bollinger_lband()
df['gold_atr'] = ta.volatility.AverageTrueRange(df['gold_high'], df['gold_low'], df['gold_close']).average_true_range()
df = df.dropna().reset_index(drop=True)
df[[c for c in df.columns if 'rsi' in c or 'macd' in c or 'bb' in c or 'atr' in c]].head()

,gold_rsi,gold_macd,gold_macd_signal,gold_macd_diff,gold_bb_high,gold_bb_low,gold_atr
0,35.687378,-5.076069,-1.593247,-3.482822,1230.738348,1163.951667,12.621707
1,39.450137,-5.837215,-2.442041,-3.395174,1231.415286,1161.684726,12.313008
2,42.214664,-6.039960,-3.161625,-2.878335,1231.662736,1160.847286,11.976372
3,47.730663,-5.427985,-3.614897,-1.813089,1231.584430,1160.275592,12.156631
4,44.507145,-5.381244,-3.968166,-1.413078,1228.744425,1159.265597,12.124011


## 3. Macro %Change Features

สร้างฟีเจอร์ %change ของ DXY, VIX, Yield, SP500, Oil

In [14]:
# --- สร้าง Macro %Change Features ---
for col in ['dxy_close','vix_close','yield_close','sp500_close','oil_close']:
    df[f'{col}_pct'] = df[col].pct_change() * 100
df = df.dropna().reset_index(drop=True)

df[[c for c in df.columns if '_pct' in c]].head()

,dxy_close_pct,vix_close_pct,yield_close_pct,sp500_close_pct,oil_close_pct
0,-1.048691,7.600281,-0.874268,-0.647451,-1.674280
1,-0.136416,-5.362980,1.511970,0.041835,3.439973
2,-0.546387,-8.638562,2.523784,1.204242,2.144997
3,0.348655,-2.798789,-3.833738,0.173863,-1.074393
4,-0.010531,7.237349,0.083932,-0.699430,-1.332897


## 4. Target Variables

สร้างตัวแปรเป้าหมายสำหรับงาน Regression (return_tomorrow) และ Classification (direction)

In [15]:
# Regression target: gold return tomorrow
df['return_tomorrow'] = df['gold_close'].pct_change().shift(-1) * 100

# Classification target: direction
def label(ret):
    if ret > 0.5:
        return 'Up'
    elif ret < -0.5:
        return 'Down'
    else:
        return 'Side'
df['direction'] = df['return_tomorrow'].apply(label)
df[['return_tomorrow', 'direction']].head(10)

,return_tomorrow,direction
0,0.349480,Side
1,0.747467,Up
2,-0.522717,Down
3,-0.093226,Side
4,0.551408,Up
5,-0.404965,Side
6,-0.347308,Side
7,2.133626,Up
8,0.000000,Side
9,-1.481486,Down
